In [3]:
import pandas as pd
import numpy as np
import json

In [2]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)

In [4]:
citation

{'doi': 'https://doi.org/10.1016/j.cell.2021.12.018',
 'citation': 'Guilliams, M., Bonnardel, J., Haest, B., Vanderborght, B., Wagner, C., Remmerie, A., Bujko, A., Martens, L., Thoné, T., Browaeys, R. and De Ponti, F.F., 2022. Spatial proteogenomics reveals distinct and evolutionarily conserved hepatic macrophage niches. Cell, 185(2), pp.379-396.',
 'tables': ['https://www.cell.com/cms/10.1016/j.cell.2021.12.018/attachment/8fa3e327-5cc5-4937-ad8d-0eeef0eb366a/mmc1.xlsx']}

## Define the metadata

In [5]:
organism = "homo_sapiens"
cell_source = "liver"
cell_state = None
table_name = "clean_degs.xlsx" # local name

## Read in file

In [26]:
excel = pd.read_excel(table_name, sheet_name = None)

ct = {i: i.split(' - ')[-1] for i in excel.keys()}

# stacks the sheets together and makes a new column "cell_type" from the sheet name
df = pd.concat(
    excel, keys=list(excel.keys())
    ).reset_index(0).rename(
        columns={"level_0": "celltype_id", "Unnamed: 0": "gene"}
        )
# # rename the cell types to be human readable
df["celltype"] = df["celltype_id"].map(ct)

df['gene'] = [g.upper() for g in df['gene']]

# Filter to Human celltypes
df = df[["Human" in ctid for ctid in df['celltype_id']]]

# Clean celltype id names
df['celltype_id'] = [ct.replace("Human ", "").replace(" DEGs", "") for ct in df['celltype_id']]

## Perform the filtering

In [27]:
col_pval = "proba_not_de"
col_lfc = "lfc_mean"
col_gene = "gene"
col_ct = "celltype_id"
col_rank = "non_zeros_proportion1"

In [ ]:
min_mean = 15
max_pval = 0.05
min_lfc = 1.5
max_gene_shares = 10
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [29]:
markers[col_ct].value_counts()

Circ. NK NKT cell    20
cDC2                 20
cDC1                 20
Macrophage           20
pDC                  20
Monocyte             20
Mig. cDC             20
Endothelial          20
Fibroblast           20
B cell               20
Plasma cell          20
Cholangiocyte        20
Neutrophil           20
Basophil             20
Res. NK cell         20
T cell               20
Hepatocyte           20
Name: celltype_id, dtype: int64

In [30]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

celltype_id
T cell               0.718529
B cell               0.738596
Basophil             0.745244
Res. NK cell         0.751515
Circ. NK NKT cell    0.784416
Cholangiocyte        0.813546
Fibroblast           0.834328
Neutrophil           0.835871
Endothelial          0.885674
Plasma cell          0.901156
cDC2                 0.928129
Macrophage           0.933522
pDC                  0.943421
Mig. cDC             0.944872
Monocyte             0.947149
Hepatocyte           0.951014
cDC1                 0.982246
Name: non_zeros_proportion1, dtype: float64

## Convert to evidence json

In [31]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
save = df.to_dict(orient = "records")

In [32]:
save

[{'cell_type_label': 'Circ. NK NKT cell',
  'gene': 'GPR56',
  'organism': 'homo_sapiens',
  'cell_source': 'liver',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'MEF2C',
  'organism': 'homo_sapiens',
  'cell_source': 'liver',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'LINC00926',
  'organism': 'homo_sapiens',
  'cell_source': 'liver',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'HLA-DMB',
  'organism': 'homo_sapiens',
  'cell_source': 'liver',
  'cell_state': None},
 {'cell_type_label': 'Circ. NK NKT cell',
  'gene': 'TXK',
  'organism': 'homo_sapiens',
  'cell_source': 'liver',
  'cell_state': None},
 {'cell_type_label': 'T cell',
  'gene': 'GZMM',
  'organism': 'homo_sapiens',
  'cell_source': 'liver',
  'cell_state': None},
 {'cell_type_label': 'Res. NK cell',
  'gene': 'IFITM1',
  'organism': 'homo_sapiens',
  'cell_source': 'liver',
  'cell_state': None},
 {'cell_type_label': 'Res. NK cell',
  'gene': 'PTGDR',
  'organi

## Save evidence

In [73]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 